# 18 — DEV oracle diagnostic: SigLIP natural probe + O1-R

Notebook Gate 1 này chỉ chạy sau khi cả sáu worker notebook 17 đã `COMPLETE`.
Nó kiểm lại marker/hash, tải đúng 202 ảnh oracle từ Volume nghiên cứu, tạo delta
SigLIP và SQLite snapshot **tạm**, rồi đo:

- **B0:** BTC-only;
- **P1-N-SIGLIP:** thêm 202 frame policy-addressable và chỉ có SigLIP để đo một
  natural-retrieval probe chẩn đoán;
- **O1-R-FINAL:** force-rank answer frame ở final candidate list để đo một ceiling,
  không phải kết quả production.

`P1-N-SIGLIP` không phải O1-T: O1-T temporal coverage đã được notebook 11 đo.
Nó cũng không phải O1-S vì chưa quét inclusive GT interval để chọn best-source,
chưa có OCR/caption/BGE parity và chưa lưu rank/score theo từng stage. Vì vậy
notebook này luôn ghi `decision_authority=diagnostic_only`; không được dùng probe
SigLIP-only để dừng augmentation hoặc chọn production policy.

Notebook không đọc held-out, không ghi `keyframes-btc`, không ingest hoặc đổi
snapshot production. Bật GPU cho bước SigLIP. Attach Kaggle Dataset
`mfe-data-v2`; notebook chấp nhận cả ZIP nguyên vẹn lẫn layout Kaggle đã bung.


In [ ]:
from pathlib import Path

RUN_ID = "oracle-eval-quang-v1"
MATERIALIZE_RUN_ID = "oracle-materialize-v1"
WORKERS = ("oracle1", "oracle2", "oracle3", "oracle4", "oracle5", "oracle6")
EXPECTED_VIDEOS = 90
EXPECTED_CANDIDATES = 202
EXPECTED_TARGETS = 219
EXPECTED_ORACLE_CANDIDATES_SHA256 = "6086f5aa35cdd43b9a4e7b0e19371dfad1d95fa0d13c32ca0259ca8278a2af5b"
EXPECTED_MATERIALIZER_CONFIG_HASH = "0aea6f3311f2697d4cc41c7bee787ea9011a44d30dd0a903cc63a188f21738c6"

RESEARCH_VOLUME = "keyframe-augmentation-research"
UPSTREAM_ROOT = "/research/keyframe-augmentation/transnetv2-btc-b1-v1"
MATERIALIZE_ROOT = f"{UPSTREAM_ROOT}/gate1-oracle-materialize-v1"
OUTPUT_ROOT = f"{UPSTREAM_ROOT}/gate1-oracle-eval-quang-v1"

PUBLISH_RESEARCH_REPORT = True
CONFIRM_PUBLISH = RUN_ID
if PUBLISH_RESEARCH_REPORT and CONFIRM_PUBLISH != RUN_ID:
    raise RuntimeError("Research publish lock is closed")

WORK = Path("/kaggle/working/oracle-eval-quang-v1")
REMOTE = WORK / "remote"
IMAGES = WORK / "images"
BASE = WORK / "base"
DELTA = WORK / "oracle-delta"
REPORTS = WORK / "reports"
for path in (REMOTE, IMAGES, BASE, DELTA, REPORTS):
    path.mkdir(parents=True, exist_ok=True)


## 1 — Clone repo, cài môi trường và mở Volume nghiên cứu

In [ ]:
import gc
import json
import os
import subprocess
import sys

REPO = Path("/kaggle/working/mfe")
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    github_token = secrets.get_secret("GITHUB_TOKEN")
    os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
    os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
except Exception as error:
    raise RuntimeError("Thiếu hoặc chưa bật quyền Kaggle Secrets GitHub/Modal") from error

askpass = Path("/kaggle/working/mfe-git-askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_REF], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO)], env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[online]", "modal"],
    check=True,
)
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoModel, AutoProcessor

from mfe.offline.augmentation.evaluation import oracle_gate, run_force_rank_variant, run_overlay_variant
from mfe.offline.augmentation.io import canonical_hash, read_jsonl, sha256_file, write_json_atomic, write_jsonl_atomic
from mfe.offline.augmentation.oracle_artifacts import (
    OracleArtifactCandidate,
    OracleArtifactTarget,
    oracle_candidate_for_snapshot,
    validate_oracle_completion_marker,
)
from mfe.offline.augmentation.snapshot import build_relational_snapshot
from mfe.offline.bundle import freeze_environment
from mfe.offline.stages import _siglip_feature_tensor
from mfe.offline.transport import ModalVolumeTransport

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 18 cần GPU cho SigLIP: bật Accelerator GPU rồi Run All lại")

config_hash = EXPECTED_MATERIALIZER_CONFIG_HASH
repo_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
research = ModalVolumeTransport(RESEARCH_VOLUME, create_if_missing=False, version=2)
print({
    "run_id": RUN_ID,
    "repo_ref": REPO_REF,
    "repo_commit": repo_commit,
    "gpu": torch.cuda.get_device_name(0),
    "btc_mode": "NO_ACCESS",
    "materializer_config_hash": config_hash,
})


## 2 — Fail closed nếu sáu worker hoặc marker/hash chưa hoàn tất

In [ ]:
manifests = []
for worker in WORKERS:
    remote = f"{MATERIALIZE_ROOT}/worker-manifests/{worker}.json"
    local = REMOTE / "manifests" / f"{worker}.json"
    research.get(remote, local)
    manifest = json.loads(local.read_text(encoding="utf-8"))
    payload = dict(manifest)
    claimed = payload.pop("manifest_payload_hash", None)
    if claimed != canonical_hash(payload):
        raise RuntimeError(f"{worker}: manifest payload hash mismatch")
    expected = {
        "status": "COMPLETE",
        "run_id": MATERIALIZE_RUN_ID,
        "worker": worker,
        "config_hash": config_hash,
        "oracle_candidates_sha256": EXPECTED_ORACLE_CANDIDATES_SHA256,
        "btc_mutated": False,
    }
    if any(manifest.get(key) != value for key, value in expected.items()):
        raise RuntimeError(f"{worker}: manifest lineage/status invalid")
    if manifest.get("assigned_count") != manifest.get("verified_complete_count"):
        raise RuntimeError(f"{worker}: worker is incomplete")
    if manifest.get("failures_this_session"):
        raise RuntimeError(f"{worker}: manifest still contains failures")
    manifests.append(manifest)

assigned = [video_id for manifest in manifests for video_id in manifest["assigned"]]
if len(assigned) != EXPECTED_VIDEOS or len(set(assigned)) != EXPECTED_VIDEOS:
    raise RuntimeError("Six worker assignments are not an exact disjoint 90-video set")

resolved_candidates = []
resolved_targets = []
image_by_id = {}
for ordinal, manifest in enumerate(manifests, 1):
    worker = manifest["worker"]
    for video_id in manifest["assigned"]:
        marker_remote = f"{MATERIALIZE_ROOT}/markers/{worker}/{video_id}.json"
        marker_local = REMOTE / "markers" / worker / f"{video_id}.json"
        research.get(marker_remote, marker_local)
        marker = json.loads(marker_local.read_text(encoding="utf-8"))
        artifacts = marker.get("artifacts", {})
        observed = {path: research.sha256(path) for path in artifacts}
        if not validate_oracle_completion_marker(
            marker,
            run_id=MATERIALIZE_RUN_ID,
            worker=worker,
            video_id=video_id,
            config_hash=config_hash,
            oracle_candidates_sha256=EXPECTED_ORACLE_CANDIDATES_SHA256,
            observed_remote_hashes=observed,
        ):
            raise RuntimeError(f"{worker}/{video_id}: marker or remote artifact hash invalid")

        candidate_remote = f"{MATERIALIZE_ROOT}/resolved-candidates/{video_id}.jsonl"
        target_remote = f"{MATERIALIZE_ROOT}/resolved-targets/{video_id}.jsonl"
        candidate_local = REMOTE / "resolved-candidates" / f"{video_id}.jsonl"
        target_local = REMOTE / "resolved-targets" / f"{video_id}.jsonl"
        research.get(candidate_remote, candidate_local)
        research.get(target_remote, target_local)
        candidates = [OracleArtifactCandidate.model_validate(row) for row in read_jsonl(candidate_local)]
        targets = [OracleArtifactTarget.model_validate(row) for row in read_jsonl(target_local)]
        resolved_candidates.extend(candidates)
        resolved_targets.extend(targets)

        image_remotes = sorted(path for path in artifacts if f"/images/{video_id}/" in path)
        if len(image_remotes) != len(candidates):
            raise RuntimeError(f"{video_id}: image/candidate count mismatch")
        for candidate in candidates:
            suffix = f"/images/{video_id}/frame_{candidate.frame_idx:06d}.jpg"
            matches = [path for path in image_remotes if path.endswith(suffix)]
            if len(matches) != 1:
                raise RuntimeError(f"{candidate.keyframe_id}: missing or ambiguous image")
            local = IMAGES / video_id / Path(matches[0]).name
            research.get(matches[0], local)
            if sha256_file(local) != artifacts[matches[0]]:
                raise RuntimeError(f"{candidate.keyframe_id}: downloaded image hash mismatch")
            image_by_id[candidate.keyframe_id] = local
    print(f"verified worker {ordinal}/{len(WORKERS)}: {worker}", flush=True)

candidate_ids = [row.keyframe_id for row in resolved_candidates]
target_pairs = [(row.query_id, row.event_index) for row in resolved_targets]
if len(candidate_ids) != EXPECTED_CANDIDATES or len(set(candidate_ids)) != EXPECTED_CANDIDATES:
    raise RuntimeError("Resolved oracle candidates are not the expected 202 unique frames")
if len(target_pairs) != EXPECTED_TARGETS or len(set(target_pairs)) != EXPECTED_TARGETS:
    raise RuntimeError("Resolved oracle targets are not the expected 219 unique query-events")
if set(image_by_id) != set(candidate_ids):
    raise RuntimeError("Image identities do not exactly match oracle candidates")
if any(row.pts_time_source != "decoder_pts" for row in resolved_candidates):
    raise RuntimeError("Oracle candidate PTS is not decoder-authoritative")
if any(row.config_hash != config_hash for row in resolved_candidates):
    raise RuntimeError("Oracle candidate config hash drift")

resolved_candidates.sort(key=lambda row: row.keyframe_id)
resolved_targets.sort(key=lambda row: (row.query_id, row.event_index))
CANDIDATES_PATH = write_jsonl_atomic(WORK / "oracle-candidates-resolved.jsonl", resolved_candidates)
TARGETS_PATH = write_jsonl_atomic(WORK / "oracle-targets-resolved.jsonl", resolved_targets)
print({"workers": 6, "videos": len(set(assigned)), "candidates": len(resolved_candidates), "targets": len(resolved_targets), "pts": "decoder_pts", "remote_integrity": "PASS"})


## 3 — Nhận diện base store `mfe-data-v2` đã attach

In [ ]:
import sqlite3
import zipfile

input_root = Path("/kaggle/input")
required_prefix = "stores/flat/data/vectors/"
required_db = "stores/btc_b1/relational/app.db"
expanded_siglip = sorted(input_root.rglob("stores/flat/data/vectors/siglip_vectors.npz"))
expanded_db = sorted(input_root.rglob(required_db))

if len(expanded_siglip) == 1 and len(expanded_db) == 1:
    dataset_root = expanded_siglip[0].parents[4]
    if expanded_db[0] != dataset_root / required_db:
        raise RuntimeError("Flat vectors và app.db không thuộc cùng một mfe-data-v2 Dataset")
    BASE_FLAT = dataset_root / "stores/flat"
    BASE_DB = expanded_db[0]
    vector_members = sorted((BASE_FLAT / "data/vectors").glob("*"))
    base_source = {"mode": "expanded_dataset", "root": str(dataset_root)}
else:
    zip_candidates = sorted(input_root.rglob("mfe-data-v2.zip"))
    if len(zip_candidates) != 1:
        raise RuntimeError(
            "Attach đúng một mfe-data-v2: cần layout stores/flat + stores/btc_b1 "
            f"hoặc file mfe-data-v2.zip; zip={zip_candidates}, siglip={expanded_siglip}, db={expanded_db}"
        )
    archive = zip_candidates[0]
    with zipfile.ZipFile(archive) as zipped:
        names = set(zipped.namelist())
        members = sorted(name for name in names if name.startswith(required_prefix) and not name.endswith("/"))
        if "stores/flat/data/vectors/siglip_vectors.npz" not in members or required_db not in names:
            raise RuntimeError("mfe-data-v2.zip thiếu base flat SigLIP hoặc app.db")
        for member in [*members, required_db]:
            target = BASE / member
            if target.is_file() and target.stat().st_size == zipped.getinfo(member).file_size:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zipped.open(member) as source, target.open("wb") as destination:
                while chunk := source.read(8 * 1024 * 1024):
                    destination.write(chunk)
    BASE_FLAT = BASE / "stores/flat"
    BASE_DB = BASE / required_db
    vector_members = sorted((BASE_FLAT / "data/vectors").glob("*"))
    base_source = {"mode": "zip", "archive": str(archive)}

if not BASE_DB.is_file() or not (BASE_FLAT / "data/vectors/siglip_vectors.npz").is_file():
    raise RuntimeError("Base discovery/extraction incomplete")
print({**base_source, "flat_files": len(vector_members), "base_db_mb": round(BASE_DB.stat().st_size / 1e6, 1)})


## 4 — Tạo delta SigLIP cho đúng 202 ảnh oracle

In [ ]:
checkpoint = "google/siglip-so400m-patch14-384"
checkpoint_revision = "9fdffc58afc957d1a03a25b10dba0329ab15c2a3"
batch_size = 16
snapshot_dir = Path(snapshot_download(checkpoint, revision=checkpoint_revision))
model_file_hashes = {
    str(path.relative_to(snapshot_dir)): sha256_file(path)
    for path in sorted(snapshot_dir.rglob("*"))
    if path.is_file()
    and (path.suffix in {".json", ".safetensors", ".txt"} or path.name.startswith("tokenizer"))
}
model_snapshot_hash = canonical_hash(model_file_hashes)
processor = AutoProcessor.from_pretrained(snapshot_dir, use_fast=True)
model = AutoModel.from_pretrained(snapshot_dir).eval().cuda()

ids = [row.keyframe_id for row in resolved_candidates]
vectors = []
for start in range(0, len(ids), batch_size):
    batch_ids = ids[start : start + batch_size]
    opened = []
    try:
        for keyframe_id in batch_ids:
            opened.append(Image.open(image_by_id[keyframe_id]).convert("RGB"))
        inputs = processor(images=opened, return_tensors="pt").to("cuda")
        with torch.inference_mode():
            features = _siglip_feature_tensor(model.get_image_features(**inputs))
        vectors.append(features.float().cpu().numpy())
    finally:
        for image in opened:
            image.close()
    print(f"SigLIP {min(start + batch_size, len(ids))}/{len(ids)}", flush=True)

matrix = np.concatenate(vectors, axis=0).astype(np.float32, copy=False)
if matrix.shape != (EXPECTED_CANDIDATES, 1152) or not np.isfinite(matrix).all():
    raise RuntimeError(f"Invalid oracle SigLIP matrix: {matrix.shape}")
vector_dir = DELTA / "data" / "vectors"
vector_dir.mkdir(parents=True, exist_ok=True)
SIGLIP_PATH = vector_dir / "siglip_vectors.npz"
np.savez_compressed(SIGLIP_PATH, ids=np.asarray(ids), vectors=matrix)
del model, processor, inputs, features, vectors, matrix
gc.collect()
torch.cuda.empty_cache()
print({
    "siglip_rows": len(ids),
    "dim": 1152,
    "checkpoint": checkpoint,
    "checkpoint_revision": checkpoint_revision,
    "model_snapshot_hash": model_snapshot_hash,
    "processor": "SiglipImageProcessor(use_fast=True)",
    "sha256": sha256_file(SIGLIP_PATH),
    "feature_parity": "BLOCKED_NO_NUMERIC_CANARY_OCR_CAPTION_BGE_OR_STAGE_FINGERPRINT",
})


## 5 — Tạo SQLite snapshot tạm, không sửa `app.db` gốc

In [ ]:
candidate_video_ids = sorted({row.video_id for row in resolved_candidates})
placeholders = ",".join("?" for _ in candidate_video_ids)
with sqlite3.connect(f"file:{BASE_DB}?mode=ro", uri=True) as connection:
    fps_rows = connection.execute(
        f"SELECT video_id, fps FROM videos WHERE video_id IN ({placeholders})",
        candidate_video_ids,
    ).fetchall()
literal_fps_by_video = {str(video_id): float(fps) for video_id, fps in fps_rows}
if set(literal_fps_by_video) != set(candidate_video_ids):
    missing = sorted(set(candidate_video_ids) - set(literal_fps_by_video))
    raise RuntimeError(f"Base DB thiếu FPS cho oracle videos: {missing[:5]}")

snapshot_candidates = [
    oracle_candidate_for_snapshot(
        candidate,
        literal_fps=literal_fps_by_video[candidate.video_id],
    )
    for candidate in resolved_candidates
]

ORACLE_DB = WORK / "oracle-relational" / "app.db"
snapshot_report = build_relational_snapshot(
    BASE_DB,
    ORACLE_DB,
    snapshot_candidates,
    overwrite=True,
    allow_preexisting_mapping_problems=True,
)
if snapshot_report["delta_rows"] != EXPECTED_CANDIDATES:
    raise RuntimeError("Oracle relational snapshot has an incomplete delta")
print(json.dumps({
    **snapshot_report,
    "oracle_artifact_pts": "decoder_pts",
    "snapshot_pts": "synthetic_literal_fps_interval",
    "frame_from_pts": "floor",
    "baseline_mapping_problems_preserved": True,
}, ensure_ascii=False, indent=2))


## 6 — Đo DEV B0, natural SigLIP probe và O1-R final-candidate ceiling

Bước này chạy đúng DEV queryset đang khóa trên nhánh Quang ba lượt. Natural probe
chỉ có quyền chẩn đoán vì delta thiếu full feature parity. O1-R chỉ cho biết ceiling
tại final candidate list nếu answer frame được ép lên đầu. Notebook ghi hash và số
query thực tế, không mặc định con số từ nhánh khác.


In [ ]:
EVAL_CONFIG = REPO / "config/btc_b1.yaml"
QUERYSET = REPO / "eval/queryset_dev.jsonl"
eval_query_ids = {str(row["query_id"]) for row in read_jsonl(QUERYSET)}
targets_in_eval = sum(row.query_id in eval_query_ids for row in resolved_targets)

b0 = run_overlay_variant(
    name="B0",
    config_path=str(EVAL_CONFIG),
    queryset_path=str(QUERYSET),
    relational_db=str(BASE_DB),
    base_flat_root=str(BASE_FLAT),
    delta_flat_root=str(DELTA),
    include_ids=[],
    latency_repeats=1,
)
natural_probe = run_overlay_variant(
    name="P1-N-SIGLIP",
    config_path=str(EVAL_CONFIG),
    queryset_path=str(QUERYSET),
    relational_db=str(ORACLE_DB),
    base_flat_root=str(BASE_FLAT),
    delta_flat_root=str(DELTA),
    include_ids=ids,
    latency_repeats=1,
)
o1r = run_force_rank_variant(
    name="O1-R-FINAL",
    config_path=str(EVAL_CONFIG),
    queryset_path=str(QUERYSET),
    relational_db=str(ORACLE_DB),
    base_flat_root=str(BASE_FLAT),
    targets=resolved_targets,
)
probe_threshold = oracle_gate(
    b0,
    natural_probe,
    min_mean_delta=0.005,
    min_rescues=2,
)

def variant_payload(run):
    return {
        "name": run.name,
        "query_ids": run.query_ids,
        "query_types": run.query_types,
        "scores": run.scores,
        "latency_ms": run.latency_ms,
        "mean_score": run.mean_score,
        "p95_latency_ms": run.p95_latency_ms,
        "by_type": run.by_type,
    }

report = {
    "status": "ORACLE_DIAGNOSTIC_COMPLETE",
    "decision_authority": "diagnostic_only",
    "split": "dev",
    "heldout_accessed": False,
    "run_id": RUN_ID,
    "materialize_run_id": MATERIALIZE_RUN_ID,
    "config_hash": config_hash,
    "repo_commit": repo_commit,
    "oracle_candidates": len(resolved_candidates),
    "oracle_targets": len(resolved_targets),
    "evaluation_queries": len(eval_query_ids),
    "oracle_targets_in_evaluation_split": targets_in_eval,
    "siglip_sha256": sha256_file(SIGLIP_PATH),
    "snapshot_sha256": sha256_file(ORACLE_DB),
    "input_hashes": {
        "base_siglip": sha256_file(BASE_FLAT / "data/vectors/siglip_vectors.npz"),
        "base_relational_db": sha256_file(BASE_DB),
        "eval_config": sha256_file(EVAL_CONFIG),
        "dev_queryset": sha256_file(QUERYSET),
        "oracle_candidates_resolved": sha256_file(CANDIDATES_PATH),
        "oracle_targets_resolved": sha256_file(TARGETS_PATH),
    },
    "variants": {run.name: variant_payload(run) for run in (b0, natural_probe, o1r)},
    "natural_probe_threshold_diagnostic_only": probe_threshold,
    "oracle_status": {
        "O1-T": "MEASURED_BY_NOTEBOOK_11_TEMPORAL_COVERAGE",
        "O1-S": "BLOCKED_NO_FULL_INTERVAL_BEST_SOURCE_FEATURE_PARITY_OR_STAGE_RANKS",
        "O1-R": "MEASURED_FINAL_CANDIDATE_FORCE_RANK_CEILING_ON_SUPPORTED_DEV_TARGETS",
    },
    "pts_contract": {
        "oracle_artifact": "decoder_pts",
        "relational_snapshot": "synthetic_literal_fps_interval",
        "frame_from_pts": "floor",
        "decoder_pts_preserved": True,
    },
    "feature_parity_status": "BLOCKED_SIGLIP_ONLY_NO_NUMERIC_CANARY_OCR_CAPTION_BGE_OR_DOWNSTREAM_FINGERPRINT",
    "failure_stage_status": "BLOCKED_MISSING_PER_EVENT_STAGE_RANK_SCORE_TRACES",
    "model": {
        "checkpoint": checkpoint,
        "revision": checkpoint_revision,
        "snapshot_hash": model_snapshot_hash,
        "processor": "SiglipImageProcessor(use_fast=True)",
    },
    "environment": freeze_environment(),
    "input_command": "Kaggle Save Version / Run All: notebook 18 DEV oracle diagnostic",
    "prohibited_inference": "Natural SigLIP-only probe cannot authorize Go/No-Go or establish O1-S.",
    "o1r_force_stage": "post_cascade_final_candidate_list_before_submission_mapping",
    "o1r_is_ceiling_only": True,
    "btc_mutated": False,
}
report["report_payload_hash"] = canonical_hash(report)
REPORT_PATH = write_json_atomic(REPORTS / "oracle-eval-report.json", report)
print(json.dumps({
    "status": report["status"],
    "B0_mean": b0.mean_score,
    "P1_N_SIGLIP_mean": natural_probe.mean_score,
    "O1_R_FINAL_mean": o1r.mean_score,
    "probe_threshold_diagnostic_only": probe_threshold,
    "report": str(REPORT_PATH),
}, ensure_ascii=False, indent=2))


## 7 — Publish report và delta nghiên cứu; BTC vẫn bất biến

In [ ]:
if PUBLISH_RESEARCH_REPORT:
    uploads = [
        (REPORT_PATH, f"{OUTPUT_ROOT}/oracle-eval-report.json"),
        (CANDIDATES_PATH, f"{OUTPUT_ROOT}/oracle-candidates-resolved.jsonl"),
        (TARGETS_PATH, f"{OUTPUT_ROOT}/oracle-targets-resolved.jsonl"),
        (SIGLIP_PATH, f"{OUTPUT_ROOT}/delta/data/vectors/siglip_vectors.npz"),
    ]
    for local, remote in uploads:
        research.put(local, remote)
        if research.sha256(remote) != sha256_file(local):
            raise RuntimeError(f"Research upload hash mismatch: {remote}")
    print({"published": OUTPUT_ROOT, "files": len(uploads), "btc_mutated": False})
else:
    print({"published": False, "local_report": str(REPORT_PATH), "btc_mutated": False})


## Cách đọc kết quả

- Không dùng `natural_probe_threshold_diagnostic_only.go` để dừng hay mở production.
- `P1-N-SIGLIP` chỉ cho biết 202 frame đã materialize có tín hiệu retrieval tự nhiên
  trong một overlay thiếu OCR/caption/BGE; nó không phải O1-T hay O1-S.
- `O1-R-FINAL` chỉ là ceiling tại final candidate list; không phải selector uplift.
- O1-S và failure-stage diagnosis vẫn BLOCKED cho tới khi có full-interval feature
  scan, full feature parity và rank/score trace theo retrieval/fusion/reranker.
- Gửi nguyên object kết quả cuối và file `oracle-eval-report.json` cho Lead review.
